### GOLD TESTING - DIMENSION MENU ITEMS (SCD2)

#### Purpose
- Validate `coffee.gold.dim_menu_items` against `coffee.silver.menu_items`
- Ensure SCD Type 2 logic is working correctly
- Verify current records using `__END_AT IS NULL`
- Validate referential integrity from `fact_transaction_items` to `dim_menu_items`

#### Tests Covered
1. Silver vs Gold row count reconciliation (current rows only)
2. Null checks on business key (`item_id`)
3. SCD2 sanity check: only 1 current row per `item_id`
4. SCD2 sanity check: invalid date ranges (`__END_AT < __START_AT`)
5. Referential integrity: `fact_transaction_items.item_id` must exist in current `dim_menu_items`


In [0]:
-- TEST 1: SILVER vs GOLD CURRENT ROW COUNT
SELECT
  'dim_menu_items_count_recon' AS test_name,
  (SELECT COUNT(*) FROM coffee.silver.menu_items) AS silver_count,
  (SELECT COUNT(*) FROM coffee.gold.dim_menu_items WHERE __END_AT IS NULL) AS gold_current_count;

In [0]:
-- TEST 2: NULL CHECK ON BUSINESS KEY
SELECT
  'dim_menu_items_null_item_id' AS test_name,
  COUNT(*) AS null_key_count
FROM coffee.gold.dim_menu_items
WHERE item_id IS NULL;


In [0]:
-- TEST 3: SCD2 SANITY - ONLY 1 CURRENT ROW PER ITEM
SELECT
  'dim_menu_items_multiple_current_rows' AS test_name,
  COUNT(*) AS keys_with_multiple_current
FROM (
  SELECT item_id
  FROM coffee.gold.dim_menu_items
  WHERE __END_AT IS NULL
  GROUP BY item_id
  HAVING COUNT(*) > 1
);

In [0]:
-- TEST 4: SCD2 SANITY - INVALID DATE RANGE CHECK
SELECT
  'dim_menu_items_invalid_date_ranges' AS test_name,
  COUNT(*) AS invalid_rows
FROM coffee.gold.dim_menu_items
WHERE __END_AT IS NOT NULL AND __END_AT < __START_AT;

In [0]:
-- TEST 5: REFERENTIAL INTEGRITY - FACT TRANSACTION ITEMS -> DIM MENU ITEMS
SELECT
  'RI_fact_transaction_items_item_id' AS test_name,
  COUNT(*) AS missing_fk_count
FROM coffee.gold.fact_transaction_items fi
LEFT JOIN coffee.gold.dim_menu_items d
  ON fi.item_id = d.item_id AND d.__END_AT IS NULL
WHERE d.item_id IS NULL;